<a href="https://colab.research.google.com/github/mahdimirnami/Tensorflow-and-Keras/blob/main/CIFAR10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

تمرین: آموزش یک شبکه عصبی عمیق روی مجموعه‌داده CIFAR10
منبع مفهومی: Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow
             (Aurélien Géron) - فصل شبکه‌های عصبی عمیق (DNN Training)

این فایل شامل سه بخش است:
  بخش ۹  : ساخت شبکه پایه با ۲۰ لایه مخفی، ۱۰۰ نورون در هر لایه، He init + ELU
  بخش ۱۰ : بهبود با Batch Normalization و مقایسه سرعت/کیفیت همگرایی
  بخش ۱۱ : اعمال Regularization با Alpha Dropout و سپس MC Dropout

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import os
import time

# **آماده‌سازی داده‌ها (مشترک بین همه بخش‌ها)**

In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.cifar10.load_data()

# نرمال‌سازی پیکسل‌ها به بازه [0, 1]
X_train_full = X_train_full.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0

# برچسب‌ها به صورت آرایه دوبعدی هستند، آن‌ها را صاف می‌کنیم
y_train_full = y_train_full.flatten()
y_test = y_test.flatten()

# جدا کردن ۵۰۰۰ نمونه برای Validation
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

print("شکل داده‌های آموزشی:", X_train.shape)
print("شکل داده‌های اعتبارسنجی:", X_valid.shape)
print("شکل داده‌های تست:", X_test.shape)

N_HIDDEN_LAYERS = 20
N_NEURONS = 100
N_CLASSES = 10
INPUT_SHAPE = [32, 32, 3]

# **بخش ۹: شبکه پایه — ۲۰ لایه مخفی، He init، فعال‌سازی ELU**

In [ ]:
def build_baseline_model():
    model = keras.models.Sequential()
    model.add(keras.layers.Flatten(input_shape=INPUT_SHAPE))
    for _ in range(N_HIDDEN_LAYERS):
        model.add(keras.layers.Dense(
            N_NEURONS,
            activation="elu",
            kernel_initializer="he_normal"
        ))
    model.add(keras.layers.Dense(N_CLASSES, activation="softmax"))
    return model


def train_baseline():
    model = build_baseline_model()

    # نرخ یادگیری کوچک برای Nadam توصیه می‌شود
    optimizer = keras.optimizers.Nadam(learning_rate=5e-5)
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer=optimizer,
                  metrics=["accuracy"])

    # Early Stopping برای جلوگیری از Overfitting و صرفه‌جویی در زمان
    early_stopping_cb = keras.callbacks.EarlyStopping(
        patience=20, restore_best_weights=True
    )
    # ذخیره بهترین مدل روی دیسک
    checkpoint_cb = keras.callbacks.ModelCheckpoint(
        "cifar10_baseline_model.keras", save_best_only=True
    )
    # لاگ برای مشاهده در TensorBoard
    run_logdir = os.path.join(os.curdir, "logs", "baseline")
    tensorboard_cb = keras.callbacks.TensorBoard(run_logdir)

    history = model.fit(
        X_train, y_train, epochs=100,
        validation_data=(X_valid, y_valid),
        callbacks=[early_stopping_cb, checkpoint_cb, tensorboard_cb]
    )
    return model, history



# **بخش ۱۰: افزودن Batch Normalization و مقایسه با حالت پایه**

In [ ]:
def build_bn_model():
    model = keras.models.Sequential()
    model.add(keras.layers.Flatten(input_shape=INPUT_SHAPE))
    model.add(keras.layers.BatchNormalization())  # BN روی ورودی هم می‌آید
    for _ in range(N_HIDDEN_LAYERS):
        model.add(keras.layers.Dense(N_NEURONS, kernel_initializer="he_normal",
                                      use_bias=False))  # بدون بایاس، چون BN خودش بایاس دارد
        model.add(keras.layers.BatchNormalization())
        model.add(keras.layers.Activation("elu"))
    model.add(keras.layers.Dense(N_CLASSES, activation="softmax"))
    return model


def train_bn():
    model = build_bn_model()
    optimizer = keras.optimizers.Nadam(learning_rate=5e-4)  # با BN می‌توان lr بزرگ‌تر گرفت
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer=optimizer,
                  metrics=["accuracy"])

    early_stopping_cb = keras.callbacks.EarlyStopping(
        patience=20, restore_best_weights=True
    )
    checkpoint_cb = keras.callbacks.ModelCheckpoint(
        "cifar10_bn_model.keras", save_best_only=True
    )
    run_logdir = os.path.join(os.curdir, "logs", "batch_norm")
    tensorboard_cb = keras.callbacks.TensorBoard(run_logdir)

    start = time.time()
    history = model.fit(
        X_train, y_train, epochs=100,
        validation_data=(X_valid, y_valid),
        callbacks=[early_stopping_cb, checkpoint_cb, tensorboard_cb]
    )
    print(f"زمان آموزش با Batch Normalization: {time.time() - start:.1f} ثانیه")
    return model, history


# **بخش ۱۱: SELU + AlphaDropout و سپس MC Dropout**

In [ ]:
def build_selu_alpha_dropout_model(dropout_rate=0.1):
    model = keras.models.Sequential()
    model.add(keras.layers.Flatten(input_shape=INPUT_SHAPE))
    for _ in range(N_HIDDEN_LAYERS):
        model.add(keras.layers.Dense(
            N_NEURONS,
            activation="selu",
            kernel_initializer="lecun_normal"  # مقداردهی لازم برای Self-Normalizing Nets
        ))
    model.add(keras.layers.AlphaDropout(rate=dropout_rate))
    model.add(keras.layers.Dense(N_CLASSES, activation="softmax"))
    return model


def train_selu_alpha_dropout():
    model = build_selu_alpha_dropout_model()
    optimizer = keras.optimizers.Nadam(learning_rate=5e-4)
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer=optimizer,
                  metrics=["accuracy"])

    # نکته مهم: با SELU باید داده‌ها استاندارد شوند (میانگین صفر، واریانس یک)
    # چون ما داده را بین [0,1] نرمال کردیم، بهتر است اینجا Standardize هم انجام شود:
    X_means = X_train.mean(axis=0)
    X_stds = X_train.std(axis=0) + 1e-7
    X_train_scaled = (X_train - X_means) / X_stds
    X_valid_scaled = (X_valid - X_means) / X_stds
    X_test_scaled = (X_test - X_means) / X_stds

    early_stopping_cb = keras.callbacks.EarlyStopping(
        patience=20, restore_best_weights=True
    )
    checkpoint_cb = keras.callbacks.ModelCheckpoint(
        "cifar10_selu_model.keras", save_best_only=True
    )
    run_logdir = os.path.join(os.curdir, "logs", "selu_alpha_dropout")
    tensorboard_cb = keras.callbacks.TensorBoard(run_logdir)

    history = model.fit(
        X_train_scaled, y_train, epochs=100,
        validation_data=(X_valid_scaled, y_valid),
        callbacks=[early_stopping_cb, checkpoint_cb, tensorboard_cb]
    )
    return model, history, (X_test_scaled, y_test)


class MCAlphaDropout(keras.layers.AlphaDropout):
    """
    نسخه‌ای از AlphaDropout که همیشه فعال است، حتی در زمان inference.
    این کار برای پیاده‌سازی MC Dropout لازم است.
    """
    def call(self, inputs, training=None):
        return super().call(inputs, training=True)


def mc_dropout_predict(model, X, n_samples=100):
    """
    پیش‌بینی با روش Monte Carlo Dropout:
    مدل را n_samples بار روی همان ورودی اجرا می‌کنیم (با dropout فعال)
    و میانگین احتمالات را به عنوان پیش‌بینی نهایی در نظر می‌گیریم.
    این کار علاوه بر پیش‌بینی، یک تخمین از عدم قطعیت مدل هم می‌دهد.
    """
    y_probas = np.stack([model(X, training=True) for _ in range(n_samples)])
    y_proba_mean = y_probas.mean(axis=0)
    return y_proba_mean


def build_mc_dropout_model_from(trained_model):
    """
    یک مدل جدید می‌سازد که لایه‌های AlphaDropout آن با MCAlphaDropout جایگزین شده
    بدون نیاز به آموزش مجدد (وزن‌های مدل قبلی کپی می‌شود).
    """
    mc_model = keras.models.Sequential([
        MCAlphaDropout(layer.rate) if isinstance(layer, keras.layers.AlphaDropout)
        else layer
        for layer in trained_model.layers
    ])
    mc_model.set_weights(trained_model.get_weights())
    return mc_model

# **اجرای کامل و مقایسه نهایی**

In [ ]:
if __name__ == "__main__":
    print("\n===== بخش ۹: آموزش شبکه پایه (۲۰ لایه، He+ELU) =====")
    baseline_model, baseline_history = train_baseline()
    baseline_loss, baseline_acc = baseline_model.evaluate(X_test, y_test)
    print(f"دقت مدل پایه روی تست: {baseline_acc:.4f}")

    print("\n===== بخش ۱۰: آموزش شبکه با Batch Normalization =====")
    bn_model, bn_history = train_bn()
    bn_loss, bn_acc = bn_model.evaluate(X_test, y_test)
    print(f"دقت مدل با BN روی تست: {bn_acc:.4f}")

    print("\n===== بخش ۱۱: آموزش شبکه با SELU + AlphaDropout =====")
    selu_model, selu_history, (X_test_scaled, y_test_s) = train_selu_alpha_dropout()
    selu_loss, selu_acc = selu_model.evaluate(X_test_scaled, y_test_s)
    print(f"دقت مدل SELU+AlphaDropout روی تست: {selu_acc:.4f}")

    print("\n----- ارزیابی با MC Dropout -----")
    mc_model = build_mc_dropout_model_from(selu_model)
    y_proba_mc = mc_dropout_predict(mc_model, X_test_scaled, n_samples=100)
    y_pred_mc = y_proba_mc.argmax(axis=1)
    mc_acc = (y_pred_mc == y_test_s).mean()
    print(f"دقت مدل با MC Dropout (میانگین ۱۰۰ نمونه): {mc_acc:.4f}")

    print("\n===== خلاصه مقایسه =====")
    print(f"مدل پایه (بخش ۹):            {baseline_acc:.4f}")
    print(f"با Batch Normalization (۱۰):  {bn_acc:.4f}")
    print(f"SELU + AlphaDropout (۱۱):     {selu_acc:.4f}")
    print(f"MC Dropout (۱۱):              {mc_acc:.4f}")